### Imports, constantes, session avec headers complets

In [1]:
import re
import requests
from bs4 import BeautifulSoup

ROOT_URL = "https://www.marchespublics.gov.ma"
SEARCH_PATH = "/index.php?page=entreprise.EntrepriseAdvancedSearch&searchAnnCons"

def new_session():
    s = requests.Session()
    s.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36 Edg/150.0.0.0",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
        "Accept-Language": "fr,fr-FR;q=0.9,en;q=0.8,en-GB;q=0.7,en-US;q=0.6",
        "Accept-Encoding": "gzip, deflate, br",
        "sec-ch-ua": '"Not;A=Brand";v="8", "Chromium";v="150", "Microsoft Edge";v="150"',
        "sec-ch-ua-mobile": "?0",
        "sec-ch-ua-platform": '"Windows"',
        "Upgrade-Insecure-Requests": "1",
    })
    return s

session = new_session()
print("Session initialisée.")

Session initialisée.


### Fonctions utilitaires (extraction de formulaire + parsing de liste)

In [2]:
def extract_form_fields(soup):
    data = {}
    for tag in soup.find_all(["input", "select", "textarea"]):
        name = tag.get("name")
        if not name:
            continue
        ttype = (tag.get("type") or "text").lower()
        if ttype in ("submit", "image", "reset", "button"):
            continue
        if ttype in ("checkbox", "radio"):
            if tag.has_attr("checked"):
                data[name] = tag.get("value", "on")
            continue
        if tag.name == "select":
            selected = tag.find("option", selected=True)
            data[name] = selected.get("value") if selected else ""
            continue
        data[name] = tag.get("value", "")
    return data


def extract_row_data(row):
    tds = row.find_all("td")
    if len(tds) < 7:
        return None
    detail_link = row.find("a", href=re.compile(r"page=entreprise\.EntrepriseDetailConsultation"))
    ref_consultation = org_acronyme = None
    if detail_link:
        m = re.search(r"refConsultation=(\w+)", detail_link["href"])
        o = re.search(r"orgAcronyme=(\w+)", detail_link["href"])
        ref_consultation = m.group(1) if m else None
        org_acronyme = o.group(1) if o else None
    return {
        "procedure_categorie": tds[1].get_text(" ", strip=True),
        "reference_objet": tds[2].get_text(" ", strip=True),
        "organisme": tds[3].get_text(" ", strip=True),
        "date_limite": tds[4].get_text(" ", strip=True),
        "ref_consultation": ref_consultation,
        "org_acronyme": org_acronyme,
    }

print("Fonctions utilitaires chargées.")


Fonctions utilitaires chargées.


### Récupérer la liste des appels d'offres

In [3]:
def fetch_appels_offres(session):
    resp_form = session.get(ROOT_URL + SEARCH_PATH, timeout=20)
    soup_form = BeautifulSoup(resp_form.text, "lxml")
    form_data = extract_form_fields(soup_form)

    form_data["PRADO_POSTBACK_TARGET"] = "ctl0$CONTENU_PAGE$AdvancedSearch$lancerRecherche"
    form_data["PRADO_POSTBACK_PARAMETER"] = ""

    resp_list = session.post(ROOT_URL + SEARCH_PATH, data=form_data, headers={"Referer": ROOT_URL + SEARCH_PATH}, timeout=20)
    soup_list = BeautifulSoup(resp_list.text, "lxml")

    table = soup_list.find("table", class_="table-results")
    rows = table.find_all("tr") if table else []

    results = [extract_row_data(r) for r in rows]
    return [r for r in results if r and r.get("ref_consultation")]


appels = fetch_appels_offres(session)
print(f"{len(appels)} appel(s) d'offres trouvé(s)")
for a in appels[:3]:
    print(a["ref_consultation"], "|", a["org_acronyme"], "|", a["reference_objet"][:60])

10 appel(s) d'offres trouvé(s)
1005689 | m1i | 01/2027/dm - ... Objet
                        : test DM ..
1019317 | d4q | SP4130189 - ... Objet
                        : Etudes d’ex
999886 | q1s | 04/2026/ENFI - ... Objet
                        : Entretie


### Récupérer la fiche détail d'un appel d'offres

In [4]:
def fetch_detail(session, ref_consultation, org_acronyme):
    detail_url = f"{ROOT_URL}/index.php?page=entreprise.EntrepriseDetailConsultation&refConsultation={ref_consultation}&orgAcronyme={org_acronyme}"
    resp = session.get(detail_url, headers={"Referer": ROOT_URL + SEARCH_PATH}, timeout=20)
    soup = BeautifulSoup(resp.text, "lxml")
    return resp, soup, detail_url


first = appels[0]
resp_detail, soup_detail, detail_url = fetch_detail(session, first["ref_consultation"], first["org_acronyme"])
print("Statut:", resp_detail.status_code, "| Titre:", soup_detail.title.get_text(strip=True) if soup_detail.title else None)

Statut: 200 | Titre: Marchés publics électroniques


### Téléchargement complet du DCE (fiche → identité → zip)

In [6]:
def download_dce(session, ref_consultation, org_acronyme, detail_url, nom="a", prenom="aze", email="a@gmail.com"):
    telechargement_url = f"{ROOT_URL}/index.php?page=entreprise.EntrepriseDemandeTelechargementDce&refConsultation={ref_consultation}&orgAcronyme={org_acronyme}"

    resp2 = session.get(telechargement_url, headers={"Referer": detail_url}, timeout=20)
    soup_dl = BeautifulSoup(resp2.text, "lxml")
    form_data = extract_form_fields(soup_dl)

    form_data["PRADO_POSTBACK_TARGET"] = "ctl0$CONTENU_PAGE$validateButton"
    form_data["PRADO_POSTBACK_PARAMETER"] = ""
    form_data["ctl0$CONTENU_PAGE$EntrepriseFormulaireDemande$RadioGroup"] = "ctl0$CONTENU_PAGE$EntrepriseFormulaireDemande$choixTelechargement"
    form_data["ctl0$CONTENU_PAGE$EntrepriseFormulaireDemande$accepterConditions"] = "on"
    form_data["ctl0$CONTENU_PAGE$EntrepriseFormulaireDemande$nom"] = nom
    form_data["ctl0$CONTENU_PAGE$EntrepriseFormulaireDemande$prenom"] = prenom
    form_data["ctl0$CONTENU_PAGE$EntrepriseFormulaireDemande$email"] = email
    form_data["ctl0$CONTENU_PAGE$EntrepriseFormulaireDemande$etablissementEntreprise"] = "ctl0$CONTENU_PAGE$EntrepriseFormulaireDemande$france"
    form_data["ctl0$CONTENU_PAGE$EntrepriseFormulaireDemande$pays"] = "0"

    resp3 = session.post(telechargement_url, data=form_data, headers={"Referer": telechargement_url}, timeout=20)
    soup_final = BeautifulSoup(resp3.text, "lxml")
    form_data_final = extract_form_fields(soup_final)

    form_data_final["PRADO_POSTBACK_TARGET"] = "ctl0$CONTENU_PAGE$EntrepriseDownloadDce$completeDownload"
    form_data_final["PRADO_POSTBACK_PARAMETER"] = ""

    resp4 = session.post(telechargement_url, data=form_data_final, headers={"Referer": telechargement_url}, timeout=20, allow_redirects=False)

    if resp4.status_code not in (301, 302) or not resp4.headers.get("Location"):
        return {"success": False, "reason": f"Pas de redirection (status {resp4.status_code})"}

    final_url = ROOT_URL + "/" + resp4.headers["Location"].lstrip("/")
    resp5 = session.get(
        final_url,
        headers={
            "Referer": telechargement_url,
            "Sec-Fetch-Dest": "document",
            "Sec-Fetch-Mode": "navigate",
            "Sec-Fetch-Site": "same-origin",
            "Sec-Fetch-User": "?1",
            "Cache-Control": "max-age=0",
        },
        timeout=30,
    )

    if "zip" not in resp5.headers.get("Content-Type", ""):
        return {"success": False, "reason": "Pas un zip", "status": resp5.status_code, "preview": resp5.text[:300]}

    filename = f"dce_{ref_consultation}_{org_acronyme}.zip"
    with open(filename, "wb") as f:
        f.write(resp5.content)
    return {"success": True, "filename": filename, "size": len(resp5.content)}


result = download_dce(session, first["ref_consultation"], first["org_acronyme"], detail_url)
print(result)

{'success': True, 'filename': 'dce_1005689_m1i.zip', 'size': 10047}


In [2]:
import requests
r = requests.get(
    "https://www.marchespublics.gov.ma/index.php?page=entreprise.EntrepriseAdvancedSearch&searchAnnCons",
    headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36 Edg/150.0.0.0"},
    timeout=20,
)
print(r.status_code)

200


In [2]:
import re
import requests
from bs4 import BeautifulSoup

ROOT_URL = "https://www.marchespublics.gov.ma"
SEARCH_URL = f"{ROOT_URL}/index.php?page=entreprise.EntrepriseAdvancedSearch&searchAnnCons"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36 Edg/150.0.0.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
    "Accept-Language": "fr,fr-FR;q=0.9,en;q=0.8,en-GB;q=0.7,en-US;q=0.6",
    "Accept-Encoding": "gzip, deflate, br",
    "sec-ch-ua": '"Not;A=Brand";v="8", "Chromium";v="150", "Microsoft Edge";v="150"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"Windows"',
    "Upgrade-Insecure-Requests": "1",
})

def extract_form_fields(soup):
    data = {}
    for tag in soup.find_all(["input", "select", "textarea"]):
        name = tag.get("name")
        if not name:
            continue
        ttype = (tag.get("type") or "text").lower()
        if ttype in ("submit", "image", "reset", "button"):
            continue
        if ttype in ("checkbox", "radio"):
            if tag.has_attr("checked"):
                data[name] = tag.get("value", "on")
            continue
        if tag.name == "select":
            selected = tag.find("option", selected=True)
            data[name] = selected.get("value") if selected else ""
            continue
        data[name] = tag.get("value", "")
    return data

# Étape 1 : charger le formulaire (établit la session)
resp_form = session.get(SEARCH_URL, timeout=20)
soup_form = BeautifulSoup(resp_form.text, "lxml")
print("Formulaire chargé:", resp_form.status_code)

# Étape 2 : vérifier les options du filtre "catégorie" directement sur le formulaire
# (pas besoin de soumettre la recherche pour voir les <option> d'un <select>)
select_cat = soup_form.find("select", attrs={"name": re.compile("categorie", re.IGNORECASE)})
if select_cat:
    print(f"\nChamp trouvé : {select_cat.get('name')}")
    for opt in select_cat.find_all("option"):
        print(repr(opt.get("value")), "|", opt.get_text(strip=True))
else:
    print("Aucun <select> avec 'categorie' dans le name. Recherche élargie :")
    for sel in soup_form.find_all("select"):
        print("-", sel.get("name"))

Formulaire chargé: 200

Champ trouvé : ctl0$CONTENU_PAGE$AdvancedSearch$categorie
'0' | ---Toutes les catégories---
'1' | Travaux
'2' | Fournitures
'3' | Services


In [4]:
import re
import requests
from bs4 import BeautifulSoup

ROOT_URL = "https://www.marchespublics.gov.ma"
SEARCH_URL = f"{ROOT_URL}/index.php?page=entreprise.EntrepriseAdvancedSearch&searchAnnCons"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36 Edg/150.0.0.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
    "Accept-Language": "fr,fr-FR;q=0.9,en;q=0.8,en-GB;q=0.7,en-US;q=0.6",
    "Accept-Encoding": "gzip, deflate, br",
    "sec-ch-ua": '"Not;A=Brand";v="8", "Chromium";v="150", "Microsoft Edge";v="150"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"Windows"',
    "Upgrade-Insecure-Requests": "1",
})

def extract_form_fields(soup):
    data = {}
    for tag in soup.find_all(["input", "select", "textarea"]):
        name = tag.get("name")
        if not name:
            continue
        ttype = (tag.get("type") or "text").lower()
        if ttype in ("submit", "image", "reset", "button"):
            continue
        if ttype in ("checkbox", "radio"):
            if tag.has_attr("checked"):
                data[name] = tag.get("value", "on")
            continue
        if tag.name == "select":
            selected = tag.find("option", selected=True)
            data[name] = selected.get("value") if selected else ""
            continue
        data[name] = tag.get("value", "")
    return data

# Formulaire puis recherche, dans la foulée, sans pause
resp_form = session.get(SEARCH_URL, timeout=20)
soup_form = BeautifulSoup(resp_form.text, "lxml")
print("Formulaire:", resp_form.status_code)

form_data = extract_form_fields(soup_form)
form_data["ctl0$CONTENU_PAGE$AdvancedSearch$categorie"] = "3"
form_data["PRADO_POSTBACK_TARGET"] = "ctl0$CONTENU_PAGE$AdvancedSearch$lancerRecherche"
form_data["PRADO_POSTBACK_PARAMETER"] = ""

resp_list = session.post(SEARCH_URL, data=form_data, headers={"Referer": SEARCH_URL}, timeout=20)
soup_list = BeautifulSoup(resp_list.text, "lxml")
print("Résultats:", resp_list.status_code, len(resp_list.text))

Formulaire: 200
Résultats: 200 297260


In [6]:
# Chercher tous les champs liés à la pagination
pagination_fields = {k: v for k, v in extract_form_fields(soup_list).items() if "page" in k.lower() or "pager" in k.lower()}
print("Champs de pagination trouvés :")
for k, v in pagination_fields.items():
    print(" ", k, "=", v)

# Chercher un lien/bouton "page suivante" ou numéros de page
print("\nLiens contenant 'page' ou du texte numérique court :")
for a in soup_list.find_all("a", href=True):
    txt = a.get_text(strip=True)
    if "suivant" in txt.lower() or (txt.isdigit() and len(txt) <= 3) or "page" in (a.get("href") or "").lower():
        print(" ", repr(txt), "->", a["href"])

# Chercher aussi des boutons submit liés à la pagination
print("\nBoutons submit contenant 'page' :")
for btn in soup_list.find_all("input", type="submit"):
    name = (btn.get("name") or "").lower()
    if "page" in name:
        print(" ", btn.get("name"), "| value:", btn.get("value"))

# Combien de résultats au total, et combien de lignes sur cette page
table = soup_list.find("table", class_="table-results")
nb_lignes = len(table.find_all("tr")) if table else 0
print(f"\n{nb_lignes} lignes dans la table (en-têtes inclus)")

# Chercher un texte du type "1 à 10 sur XX résultats"
import re
compteur = soup_list.find(string=re.compile(r"\d+\s*(à|sur|/)\s*\d+"))
print("Texte de comptage trouvé :", repr(compteur.strip()) if compteur else None)

Champs de pagination trouvés :
  PRADO_PAGESTATE = ...
  ctl0$CONTENU_PAGE$resultSearch$listePageSizeTop = 10
  ctl0$CONTENU_PAGE$resultSearch$numPageTop = 1
  ctl0$CONTENU_PAGE$resultSearch$tableauResultSearch$ctl1$refCons = 999886
  ctl0$CONTENU_PAGE$resultSearch$tableauResultSearch$ctl1$orgCons = q1s
  ctl0$CONTENU_PAGE$resultSearch$tableauResultSearch$ctl2$refCons = 1023743
  ctl0$CONTENU_PAGE$resultSearch$tableauResultSearch$ctl2$orgCons = w7t
  ctl0$CONTENU_PAGE$resultSearch$tableauResultSearch$ctl3$refCons = 999466
  ctl0$CONTENU_PAGE$resultSearch$tableauResultSearch$ctl3$orgCons = p8x
  ctl0$CONTENU_PAGE$resultSearch$tableauResultSearch$ctl4$refCons = 1019055
  ctl0$CONTENU_PAGE$resultSearch$tableauResultSearch$ctl4$orgCons = m1w
  ctl0$CONTENU_PAGE$resultSearch$tableauResultSearch$ctl5$refCons = 1023773
  ctl0$CONTENU_PAGE$resultSearch$tableauResultSearch$ctl5$orgCons = q9t
  ctl0$CONTENU_PAGE$resultSearch$tableauResultSearch$ctl6$refCons = 1013649
  ctl0$CONTENU_PAGE$resultSe

In [7]:
print("=== Liens/pager autour de la table ===")
table = soup_list.find("table", class_="table-results")
# Cherche le conteneur de pagination, généralement juste après la table
if table:
    pager = table.find_next_sibling()
    tries = 0
    while pager and tries < 5:
        text = pager.get_text(" ", strip=True)
        if text:
            print(f"--- Sibling {tries} ---")
            print(text[:300])
            for a in pager.find_all("a", href=True):
                print("  lien:", a.get_text(strip=True), "->", a["href"])
        pager = pager.find_next_sibling()
        tries += 1

print("\n=== Recherche globale de liens vers une page 2 ===")
for a in soup_list.find_all("a"):
    txt = a.get_text(strip=True)
    href = a.get("href", "")
    if txt == "2" or "suivant" in txt.lower() or "numPage" in href:
        print(" ", repr(txt), "->", href, "| onclick:", a.get("onclick"))

print("\n=== Boutons submit contenant 'page' (sans troncature) ===")
for btn in soup_list.find_all("input", type="submit"):
    name = (btn.get("name") or "")
    if "page" in name.lower():
        print(" ", name, "| value:", btn.get("value"))

print("\n=== Total résultats (cherche un nombre après 'résultats' ou similaire) ===")
import re
total_text = soup_list.find(string=re.compile(r"\d+\s*résultat", re.IGNORECASE))
print("Trouvé:", repr(total_text.strip()) if total_text else None)

=== Liens/pager autour de la table ===
--- Sibling 0 ---
Afficher 10 20 50 100 500 résultats / page Aller à la page / 140
  lien:  -> javascript:;//ctl0_CONTENU_PAGE_resultSearch_PagerBottom_ctl2
  lien:  -> javascript:;//ctl0_CONTENU_PAGE_resultSearch_PagerBottom_ctl3

=== Recherche globale de liens vers une page 2 ===

=== Boutons submit contenant 'page' (sans troncature) ===
  ctl0$CONTENU_PAGE$resultSearch$DefaultButtonTop | value: 
  ctl0$CONTENU_PAGE$resultSearch$DefaultButtonBottom | value: 
  ctl0$CONTENU_PAGE$afficherMonPanier | value: 

=== Total résultats (cherche un nombre après 'résultats' ou similaire) ===
Trouvé: None


In [8]:
form_data_p2 = extract_form_fields(soup_list)
form_data_p2["ctl0$CONTENU_PAGE$resultSearch$numPageTop"] = "2"
form_data_p2["PRADO_POSTBACK_TARGET"] = "ctl0$CONTENU_PAGE$resultSearch$DefaultButtonTop"
form_data_p2["PRADO_POSTBACK_PARAMETER"] = ""

resp_p2 = session.post(SEARCH_URL, data=form_data_p2, headers={"Referer": SEARCH_URL}, timeout=20)
soup_p2 = BeautifulSoup(resp_p2.text, "lxml")
print("Statut:", resp_p2.status_code)

# Vérifier qu'on a bien changé de page : comparer les refCons de page 1 vs page 2
refs_p2 = re.findall(r"tableauResultSearch\$ctl\d+\$refCons.*?=\s*(\d+)", resp_p2.text)
print("Références trouvées en page 2 :", refs_p2[:12])

# Confirmer le nouveau numéro de page affiché
confirm = soup_p2.find(string=re.compile(r"Aller à la page"))
print("Confirmation page :", repr(confirm.strip()) if confirm else None)

Statut: 200
Références trouvées en page 2 : []
Confirmation page : 'Aller à la page'


In [9]:
# Utiliser extract_form_fields (déjà fiable) plutôt qu'un regex sur le HTML brut
fields_p2 = extract_form_fields(soup_p2)
refs_p2 = {k: v for k, v in fields_p2.items() if "refCons" in k}
print("Références en page 2 :", refs_p2)

# Comparaison directe avec la page 1
fields_p1 = extract_form_fields(soup_list)
refs_p1 = {k: v for k, v in fields_p1.items() if "refCons" in k}
print("Références en page 1 :", refs_p1)

print("\nMême contenu ?", refs_p1 == refs_p2)

# Vérifier aussi le numéro de page réellement actif après la requête
num_page_actuel = fields_p2.get("ctl0$CONTENU_PAGE$resultSearch$numPageTop")
print("numPageTop actuel après la requête :", num_page_actuel)

# Regarder le texte complet autour de "Aller à la page" pour voir s'il indique la page courante
import re
context = re.search(r".{0,50}Aller à la page.{0,80}", resp_p2.text)
print("Contexte:", repr(context.group(0)) if context else None)

Références en page 2 : {'ctl0$CONTENU_PAGE$resultSearch$tableauResultSearch$ctl1$refCons': '1016110', 'ctl0$CONTENU_PAGE$resultSearch$tableauResultSearch$ctl2$refCons': '1021912', 'ctl0$CONTENU_PAGE$resultSearch$tableauResultSearch$ctl3$refCons': '1021904', 'ctl0$CONTENU_PAGE$resultSearch$tableauResultSearch$ctl4$refCons': '1021878', 'ctl0$CONTENU_PAGE$resultSearch$tableauResultSearch$ctl5$refCons': '1022621', 'ctl0$CONTENU_PAGE$resultSearch$tableauResultSearch$ctl6$refCons': '1023753', 'ctl0$CONTENU_PAGE$resultSearch$tableauResultSearch$ctl7$refCons': '1023987', 'ctl0$CONTENU_PAGE$resultSearch$tableauResultSearch$ctl8$refCons': '976822', 'ctl0$CONTENU_PAGE$resultSearch$tableauResultSearch$ctl9$refCons': '1021499', 'ctl0$CONTENU_PAGE$resultSearch$tableauResultSearch$ctl10$refCons': '1022616'}
Références en page 1 : {'ctl0$CONTENU_PAGE$resultSearch$tableauResultSearch$ctl1$refCons': '999886', 'ctl0$CONTENU_PAGE$resultSearch$tableauResultSearch$ctl2$refCons': '1023743', 'ctl0$CONTENU_PAG

In [2]:
import re
import requests
from bs4 import BeautifulSoup

ROOT_URL = "https://www.marchespublics.gov.ma"
SEARCH_URL = f"{ROOT_URL}/index.php?page=entreprise.EntrepriseAdvancedSearch&searchAnnCons"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36 Edg/150.0.0.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
    "Accept-Language": "fr,fr-FR;q=0.9,en;q=0.8,en-GB;q=0.7,en-US;q=0.6",
    "Accept-Encoding": "gzip, deflate, br",
    "sec-ch-ua": '"Not;A=Brand";v="8", "Chromium";v="150", "Microsoft Edge";v="150"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"Windows"',
    "Upgrade-Insecure-Requests": "1",
})

def extract_form_fields(soup):
    data = {}
    for tag in soup.find_all(["input", "select", "textarea"]):
        name = tag.get("name")
        if not name:
            continue
        ttype = (tag.get("type") or "text").lower()
        if ttype in ("submit", "image", "reset", "button"):
            continue
        if ttype in ("checkbox", "radio"):
            if tag.has_attr("checked"):
                data[name] = tag.get("value", "on")
            continue
        if tag.name == "select":
            selected = tag.find("option", selected=True)
            data[name] = selected.get("value") if selected else ""
            continue
        data[name] = tag.get("value", "")
    return data

# Étape 1 : charger le formulaire
resp_form = session.get(SEARCH_URL, timeout=20)
soup_form = BeautifulSoup(resp_form.text, "lxml")
print("Formulaire chargé:", resp_form.status_code, "| taille:", len(resp_form.text))

print("\n=== Cellule A : champs liés à 'domaine' ===")
domaine_related = {k: v for k, v in extract_form_fields(soup_form).items() if "domaine" in k.lower()}
for k, v in domaine_related.items():
    print(" ", k, "=", repr(v))

for btn in soup_form.find_all("input", attrs={"name": re.compile("domaine", re.IGNORECASE)}):
    print("Bouton:", btn.get("name"), "| type:", btn.get("type"), "| value:", btn.get("value"))

print("\n=== Cellule B : popups/libellés liés au domaine ===")
matches = re.findall(r'(?:popUp|window\.open)\([\'"]([^\'"]*[Dd]omaine[^\'"]*)[\'"]', resp_form.text)
print("Popups liés au domaine :", matches)

label = soup_form.find(string=re.compile(r"Domaine d.activit", re.IGNORECASE))
print("Libellé trouvé :", repr(label.strip()) if label else None)
if label:
    parent = label.find_parent()
    print(parent.prettify()[:1500])

print("\n=== Cellule C : scripts liés au domaine ===")
scripts = soup_form.find_all("script")
for s in scripts:
    if s.string and "domaine" in s.string.lower():
        print(s.string[:2000])
        print("---")

Formulaire chargé: 200 | taille: 104606

=== Cellule A : champs liés à 'domaine' ===
  ctl0$CONTENU_PAGE$AdvancedSearch$domaineActivite$idsDomaines = ''
Bouton: ctl0$CONTENU_PAGE$AdvancedSearch$domaineActivite$displayDomaine | type: submit | value: 

=== Cellule B : popups/libellés liés au domaine ===
Popups liés au domaine : ['index.php?page=commun.PopupDomainesActivites&ids=&clientId=ctl0_CONTENU_PAGE_AdvancedSearch_domaineActivite']
Libellé trouvé : "D�but domaine d'activit�s Lt-Ref"
<div class="content">
 <div class="line" id="ctl0_CONTENU_PAGE_AdvancedSearch_panelModeRechercheEntite" style="display:none;">
  <div class="intitule-150">
   <label for="orgNameAM">
    <span id="ctl0_CONTENU_PAGE_AdvancedSearch_messageModeRecherechEntite">
     Entité publique
    </span>
   </label>
   :
  </div>
  <div class="content-bloc bloc-600" style="display:none;">
   <div class="line">
    <span class="radio" title="Recherche par autocomplétion">
     <input checked="checked" class="radio" id

In [3]:
POPUP_URL = f"{ROOT_URL}/index.php?page=commun.PopupDomainesActivites&ids=&clientId=ctl0_CONTENU_PAGE_AdvancedSearch_domaineActivite"

resp_popup = session.get(POPUP_URL, headers={"Referer": SEARCH_URL}, timeout=20)
soup_popup = BeautifulSoup(resp_popup.text, "lxml")
print("Statut:", resp_popup.status_code, "| taille:", len(resp_popup.text))
print("Titre:", soup_popup.title.get_text(strip=True) if soup_popup.title else None)

# Cherche toutes les checkboxes de la popup (chaque domaine a probablement un id numérique)
checkboxes = soup_popup.find_all("input", type="checkbox")
print(f"\n{len(checkboxes)} checkbox(es) trouvée(s)")
for cb in checkboxes[:30]:
    # Cherche le libellé associé (souvent dans un <label> ou le texte juste après)
    label_text = ""
    if cb.get("id"):
        label = soup_popup.find("label", attrs={"for": cb["id"]})
        if label:
            label_text = label.get_text(strip=True)
    if not label_text:
        label_text = cb.find_parent().get_text(strip=True)[:80] if cb.find_parent() else ""
    print(" id:", cb.get("id"), "| name:", cb.get("name"), "| value:", cb.get("value"), "| libellé:", label_text)

# Sauvegarde pour inspection si besoin
with open("debug_popup_domaines.html", "w", encoding="utf-8") as f:
    f.write(resp_popup.text)
print("\nSauvegardé : debug_popup_domaines.html")

Statut: 200 | taille: 298858
Titre: Marchés publics électroniques

391 checkbox(es) trouvée(s)
 id: ctl0_CONTENU_PAGE_repeaterCategorie_ctl0_idCategorie | name: ctl0$CONTENU_PAGE$repeaterCategorie$ctl0$idCategorie | value: 1 | libellé: 
 id: ctl0_CONTENU_PAGE_repeaterCategorie_ctl0_repeaterSousCategorie_ctl0_idSection | name: ctl0$CONTENU_PAGE$repeaterCategorie$ctl0$repeaterSousCategorie$ctl0$idSection | value: 1.10 | libellé: 
 id: ctl0_CONTENU_PAGE_repeaterCategorie_ctl0_repeaterSousCategorie_ctl0_repeaterCatNiv3_ctl0_idBranche | name: ctl0$CONTENU_PAGE$repeaterCategorie$ctl0$repeaterSousCategorie$ctl0$repeaterCatNiv3$ctl0$idBranche | value: 1.10.1 | libellé: 
 id: ctl0_CONTENU_PAGE_repeaterCategorie_ctl0_repeaterSousCategorie_ctl0_repeaterCatNiv3_ctl1_idBranche | name: ctl0$CONTENU_PAGE$repeaterCategorie$ctl0$repeaterSousCategorie$ctl0$repeaterCatNiv3$ctl1$idBranche | value: 1.10.2 | libellé: 
 id: ctl0_CONTENU_PAGE_repeaterCategorie_ctl0_repeaterSousCategorie_ctl0_repeaterCatNiv3_c

In [4]:
# Chercher directement le texte des 3 domaines qui nous intéressent
targets = [
    "ingénierie",
    "conseil, audit",
    "maitrise d'ouvrage",
    "maîtrise d'ouvrage",
    "technologies de l'information",
    "télécommunications",
    "nouvelles technologies",
]

for target in targets:
    found = soup_popup.find_all(string=re.compile(re.escape(target), re.IGNORECASE))
    print(f"\n=== '{target}' : {len(found)} occurrence(s) ===")
    for f in found[:5]:
        print(" texte:", repr(f.strip()))
        # Remonte au conteneur pour trouver la checkbox associée (frère/parent proche)
        container = f.find_parent(["tr", "li", "div", "td"])
        if container:
            cb = container.find("input", type="checkbox")
            if not cb:
                # cherche dans le parent au-dessus si pas trouvé directement
                cb = container.find_parent().find("input", type="checkbox") if container.find_parent() else None
            print("   checkbox associée -> name:", cb.get("name") if cb else None, "| value:", cb.get("value") if cb else None)


=== 'ingénierie' : 1 occurrence(s) ===
 texte: 'Etudes d’ingénierie'
   checkbox associée -> name: ctl0$CONTENU_PAGE$repeaterCategorie$ctl2$repeaterSousCategorie$ctl0$idSection | value: 3.10

=== 'conseil, audit' : 1 occurrence(s) ===
 texte: 'Conseil, audit et assistance à maitrise d’ouvrage (à l’exception du domaine des nouvelles technologies)'
   checkbox associée -> name: ctl0$CONTENU_PAGE$repeaterCategorie$ctl2$repeaterSousCategorie$ctl1$idSection | value: 3.11

=== 'maitrise d'ouvrage' : 0 occurrence(s) ===

=== 'maîtrise d'ouvrage' : 0 occurrence(s) ===

=== 'technologies de l'information' : 1 occurrence(s) ===
 texte: "Services de technologies de l'information et télécommunications"
   checkbox associée -> name: ctl0$CONTENU_PAGE$repeaterCategorie$ctl2$repeaterSousCategorie$ctl9$idSection | value: 3.19

=== 'télécommunications' : 1 occurrence(s) ===
 texte: "Services de technologies de l'information et télécommunications"
   checkbox associée -> name: ctl0$CONTENU_PAGE$repeate

In [5]:
# Chercher le bouton de validation de la popup (celui qui renvoie la sélection au formulaire parent)
for btn in soup_popup.find_all("input", type="submit"):
    print("Bouton submit:", btn.get("name"), "| value:", btn.get("value"))

# Chercher aussi une éventuelle fonction JS qui construit idsDomaines à partir des cases cochées
scripts = soup_popup.find_all("script")
for s in scripts:
    if s.string and ("idsDomaines" in s.string or "valider" in s.string.lower()):
        # Affiche le morceau pertinent
        idx = s.string.lower().find("validersel") if "validersel" in s.string.lower() else s.string.find("idsDomaines")
        print(s.string[max(0,idx-200):idx+800])
        print("---")

# Vérifier si idsDomaines attend un format simple : liste de codes séparés par virgule ?
# On regarde le nom du champ caché correspondant côté formulaire principal (déjà connu) :
print("\nRappel champ cible dans le formulaire principal :")
print("ctl0$CONTENU_PAGE$AdvancedSearch$domaineActivite$idsDomaines")

Bouton submit: ctl0$CONTENU_PAGE$ctl3 | value: Annuler
Bouton submit: ctl0$CONTENU_PAGE$validateButton | value: Valider

Rappel champ cible dans le formulaire principal :
ctl0$CONTENU_PAGE$AdvancedSearch$domaineActivite$idsDomaines


In [6]:
# Test direct : soumettre la recherche principale avec les 3 domaines ciblés
form_data_test = extract_form_fields(soup_form)
form_data_test["ctl0$CONTENU_PAGE$AdvancedSearch$categorie"] = "3"  # Services
form_data_test["ctl0$CONTENU_PAGE$AdvancedSearch$domaineActivite$idsDomaines"] = "3.10,3.11,3.19"
form_data_test["PRADO_POSTBACK_TARGET"] = "ctl0$CONTENU_PAGE$AdvancedSearch$lancerRecherche"
form_data_test["PRADO_POSTBACK_PARAMETER"] = ""

resp_test = session.post(SEARCH_URL, data=form_data_test, headers={"Referer": SEARCH_URL}, timeout=20)
soup_test = BeautifulSoup(resp_test.text, "lxml")
print("Statut:", resp_test.status_code, "| taille:", len(resp_test.text))

# Vérifier si le champ domaineActivite a bien conservé la valeur après la recherche
fields_after = extract_form_fields(soup_test)
print("idsDomaines après recherche :", repr(fields_after.get("ctl0$CONTENU_PAGE$AdvancedSearch$domaineActivite$idsDomaines")))

# Vérifier si un texte affiche les domaines sélectionnés quelque part sur la page résultat
domaine_display = soup_test.find(string=re.compile(r"ingénierie|Conseil, audit|technologies de l.information", re.IGNORECASE))
print("Texte domaine affiché :", repr(domaine_display.strip()) if domaine_display else None)

# Compter les résultats obtenus
table_test = soup_test.find("table", class_="table-results")
nb_lignes = len(table_test.find_all("tr")) if table_test else 0
print("Lignes dans la table de résultats :", nb_lignes)

# Comparaison : refaire la même recherche SANS le filtre domaine, pour comparer le volume
form_data_no_filter = extract_form_fields(soup_form)
form_data_no_filter["ctl0$CONTENU_PAGE$AdvancedSearch$categorie"] = "3"
form_data_no_filter["PRADO_POSTBACK_TARGET"] = "ctl0$CONTENU_PAGE$AdvancedSearch$lancerRecherche"
form_data_no_filter["PRADO_POSTBACK_PARAMETER"] = ""

resp_no_filter = session.post(SEARCH_URL, data=form_data_no_filter, headers={"Referer": SEARCH_URL}, timeout=20)
soup_no_filter = BeautifulSoup(resp_no_filter.text, "lxml")
table_no_filter = soup_no_filter.find("table", class_="table-results")
nb_lignes_no_filter = len(table_no_filter.find_all("tr")) if table_no_filter else 0
print("\nSans filtre domaine, lignes dans la table :", nb_lignes_no_filter)

pager_test = soup_test.find(string=re.compile(r"Aller à la page"))
print("Pagination avec filtre :", repr(pager_test.strip()) if pager_test else None)

Statut: 200 | taille: 58066
idsDomaines après recherche : None
Texte domaine affiché : None
Lignes dans la table de résultats : 0

Sans filtre domaine, lignes dans la table : 12
Pagination avec filtre : None


In [7]:
print("Titre:", soup_test.title.get_text(strip=True) if soup_test.title else None)

# Chercher un message d'erreur/validation
error_el = soup_test.find(attrs={"class": re.compile(r"error|alert|warning|invalid", re.IGNORECASE)})
print("Élément d'erreur:", error_el.get_text(strip=True) if error_el else None)

# Afficher le début du texte visible de la page pour comprendre ce qui est réellement retourné
visible_text = soup_test.get_text(" ", strip=True)
print("\nDébut du texte visible de la page :")
print(visible_text[:1000])

# Vérifier s'il y a un message "aucun résultat"
no_result_msg = soup_test.find(string=re.compile(r"aucun résultat|aucune annonce|0 résultat", re.IGNORECASE))
print("\nMessage 'aucun résultat' :", repr(no_result_msg.strip()) if no_result_msg else None)

Titre: Marchés publics électroniques
Élément d'erreur: None

Début du texte visible de la page :
Marchés publics électroniques Aller au menu Aller au contenu Langue de navigation : Vous n'êtes pas authentifié PORTAIL Bienvenue S'identifier Mon panier Consultations Toutes les consultations Avec / sans retrait Avec / sans question posée Avec / sans dépôt Avec / sans message échangé Toutes les consultations Consultations clôturées Recherche avancée Annonces Consultations Recherche rapide Ex 1: Fourniture d'écrans pour micro-ordinateurs : Recherche de type OU sur chaque mot, hors mots de liaison (le, la, les, avec,pour, etc.) Ex 2 : "Fourniture d'écrans pour micro-ordinateurs" : Recherche sur l'expression exacte (car présence de guillemets) Consultations en cours Toutes les consultations Recherche avancée Bon de commande Avis d'achat en cours Autres annonces Toutes les annonces d'information Tous les extraits de PV Tous les résultats définitifs Tous les rapports d'achèvement Tous les rappo